### `Generate Key`

In [ ]:
import random
import string


def generate_key() -> str:
    length = random.randint(3, 6)
    return "".join(random.choices(string.ascii_uppercase, k=length))

key = generate_key()
print(f"Generated Key: {key}")

### `Encryption`

In [ ]:
import os


def encrypt(plain: str, key: str) -> str:
    result = ""
    key = key.upper()
    key_index = 0

    for ch in plain:
        if ch.isascii() and ch.isalpha():
            shift = ord(key[key_index % len(key)]) - ord("A")

            if ch.isupper():
                result += chr((ord(ch) - ord("A") + shift) % 26 + ord("A"))
            else:
                result += chr((ord(ch) - ord("a") + shift) % 26 + ord("a"))

            key_index += 1
        else:
            result += ch

    return result


file_path = input("Enter path of plain text file: ").strip()
key = input("Enter key value: ").strip()

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    with open(file_path, 'r', encoding='utf-8') as f:
        plain_text = f.read()
    
    encrypted_text = encrypt(plain_text, key)
    
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    directory = os.path.dirname(file_path)
    output_file = os.path.join(directory, f"{file_name}_Encrypted_VC.txt")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(encrypted_text)
    
    print(f"Encryption successful!")
    print(f"Key used: {key}")
    print(f"Original file: {file_path}")
    print(f"Encrypted file saved: {output_file}")

### `Decryption`

In [ ]:
import os


def decrypt(cipher: str, key: str) -> str:
    result = ""
    key = key.upper()
    key_index = 0

    for ch in cipher:
        if ch.isascii() and ch.isalpha():
            shift = ord(key[key_index % len(key)]) - ord("A")

            if ch.isupper():
                result += chr((ord(ch) - ord("A") - shift) % 26 + ord("A"))
            else:
                result += chr((ord(ch) - ord("a") - shift) % 26 + ord("a"))

            key_index += 1
        else:
            result += ch

    return result


file_path = input("Enter path of encrypted text file: ").strip()
key = input("Enter key value: ").strip()

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    with open(file_path, 'r', encoding='utf-8') as f:
        encrypted_text = f.read()
    
    decrypted_text = decrypt(encrypted_text, key)
    
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    directory = os.path.dirname(file_path)
    output_file = os.path.join(directory, f"{file_name}_Decrypted_VC.txt")
    
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(decrypted_text)
    
    print(f"Decryption successful!")
    print(f"Key used: {key}")
    print(f"Original file: {file_path}")
    print(f"Decrypted file saved: {output_file}")

### `Frequency Analysis Attack`

In [ ]:
import os
from collections import Counter

ENGLISH_FREQ = [
    0.08167, 0.01492, 0.02782, 0.04253, 0.12702, 0.02228, 0.02015, 0.06094,
    0.06966, 0.00153, 0.00772, 0.04025, 0.02406, 0.06749, 0.07507, 0.01929,
    0.00095, 0.05987, 0.06327, 0.09056, 0.02758, 0.00978, 0.02360, 0.00150,
    0.01974, 0.00074,
]

COMMON_BIGRAMS = [
    "TH", "HE", "IN", "ER", "AN", "RE", "ON", "AT", "EN", "ND",
    "TI", "ES", "OR", "TE", "OF", "ED", "IS", "IT", "AL", "AR",
    "ST", "TO", "NT", "NG", "SE", "HA", "AS", "OU", "IO", "LE",
]


def get_top_key_lengths(cipher_alpha: str, max_key_len: int = 20, top_n: int = 3) -> list[int]:
    ic_scores = {}
    for key_len in range(1, max_key_len + 1):
        ic_total = 0
        for i in range(key_len):
            subseq = cipher_alpha[i::key_len]
            n = len(subseq)
            if n < 2:
                continue
            freq = Counter(subseq)
            ic = sum(f * (f - 1) for f in freq.values()) / (n * (n - 1))
            ic_total += ic
        avg_ic = ic_total / key_len
        ic_scores[key_len] = avg_ic
    return sorted(ic_scores, key=ic_scores.get, reverse=True)[:top_n]


def frequency_attack_chi_square(cipher_alpha: str, key_len: int) -> str:
    key = []
    for i in range(key_len):
        subseq = cipher_alpha[i::key_len]
        n = len(subseq)
        counts = [0] * 26
        for c in subseq:
            counts[ord(c) - 65] += 1
        best_shift = 0
        best_chi_sq = float("inf")
        for shift in range(26):
            chi_sq = 0
            for c_val in range(26):
                orig_val = (c_val - shift) % 26
                expected = n * ENGLISH_FREQ[orig_val]
                observed = counts[c_val]
                if expected > 0:
                    chi_sq += ((observed - expected) ** 2) / expected
            if chi_sq < best_chi_sq:
                best_chi_sq = chi_sq
                best_shift = shift
        key.append(chr(best_shift + 65))
    return "".join(key)


def score_plaintext(text: str) -> int:
    text = text.upper()
    return sum(text.count(bg) for bg in COMMON_BIGRAMS)


def decrypt(cipher: str, key: str) -> str:
    result = []
    key_shifts = [ord(k) - 65 for k in key.upper()]
    key_len = len(key)
    key_index = 0
    for ch in cipher:
        if ch.isascii() and ch.isalpha():
            shift = key_shifts[key_index % key_len]
            if ch.isupper():
                result.append(chr((ord(ch) - 65 - shift) % 26 + 65))
            else:
                result.append(chr((ord(ch) - 97 - shift) % 26 + 97))
            key_index += 1
        else:
            result.append(ch)
    return "".join(result)


def reduce_key(key: str) -> str:
    for length in range(1, len(key) + 1):
        if len(key) % length == 0:
            pattern = key[:length]
            if pattern * (len(key) // length) == key:
                return pattern
    return key


def vigenere_attack(ciphertext: str) -> dict:
    cipher_alpha = "".join(c for c in ciphertext.upper() if c.isascii() and c.isalpha())
    if not cipher_alpha:
        return {"guessed_key": "", "guessed_plaintext": ""}

    top_lengths = get_top_key_lengths(cipher_alpha)
    best_key = ""
    best_text = ""
    best_score = -1

    for length in top_lengths:
        candidate_key = frequency_attack_chi_square(cipher_alpha, length)
        candidate_text = decrypt(ciphertext, candidate_key)
        english_score = score_plaintext("".join(c for c in candidate_text if c.isascii() and c.isalpha()))

        if english_score > best_score:
            best_score = english_score
            best_key = candidate_key
            best_text = candidate_text

    return {
        "guessed_key": reduce_key(best_key),
        "guessed_plaintext": best_text,
    }


file_path = input("Enter path of encrypted text file: ").strip()

if not os.path.exists(file_path):
    print(f"Error: File '{file_path}' not found.")
else:
    with open(file_path, 'r', encoding='utf-8') as f:
        ciphertext = f.read()
    
    attack_result = vigenere_attack(ciphertext)
    
    file_name = os.path.splitext(os.path.basename(file_path))[0]
    directory = os.path.dirname(file_path)
    output_file = os.path.join(directory, f"{file_name}_Attacked_VC.txt")

    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(attack_result['guessed_plaintext'])

    print(f"Attack successful!")
    print(f"Guessed Key: {attack_result['guessed_key']}")
    print(f"Output written to: {output_file}")